In [0]:
# =============================================================================
# BRONZE LAYER - RAW DATA INGESTION
# =============================================================================
# Purpose: Ingest raw crypto OHLCV data from CryptoCompare API
# Output: cryptocatalog.crypto_raw.bronze_ohlcv_1min
# =============================================================================

# Configuration
catalog_name = "cryptocatalog"
bronze_table = f"{catalog_name}.crypto_raw.bronze_ohlcv_1min"

print("🥉 BRONZE LAYER CONFIGURATION")
print("=" * 80)
print(f"Catalog: {catalog_name}")
print(f"Target Table: {bronze_table}")
print("=" * 80)

# Create Unity Catalog schemas if not exists
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.crypto_raw")
    spark.sql(f"ALTER SCHEMA {catalog_name}.crypto_raw SET DBPROPERTIES ('layer' = 'bronze', 'description' = 'Raw crypto data from APIs')")
    print(f"\n✅ Schema ready: {catalog_name}.crypto_raw")
except Exception as e:
    print(f"⚠️ Schema setup: {e}")

In [0]:
import requests
import json
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

def fetch_crypto_data_batch(symbol="BTC", compare="USD", limit=100):
    """
    Fetch batch of 1-minute OHLCV data from CryptoCompare API
    Returns: List of dictionaries containing OHLCV data
    """
    url = "https://min-api.cryptocompare.com/data/v2/histominute"
    params = {"fsym": symbol, "tsym": compare, "limit": limit}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('Response') == 'Success':
                return data['Data']['Data']
        return []
    except Exception as e:
        print(f"❌ Error fetching data: {e}")
        return []

print("✅ API fetch function loaded")

In [0]:
def ingest_to_bronze():
    """
    Bronze Layer: Ingest raw data with metadata
    - Fetches data from CryptoCompare API
    - Adds ingestion metadata (timestamp, source, quality flag)
    - Appends to bronze table (allows duplicates for audit trail)
    """
    print("=" * 80)
    print("🥉 BRONZE LAYER - RAW DATA INGESTION")
    print("=" * 80)
    
    # Fetch data from API
    raw_data = fetch_crypto_data_batch("BTC", "USD", limit=100)
    
    if not raw_data:
        print("⚠️ No data fetched from API")
        return None
    
    print(f"📥 Fetched {len(raw_data)} records from API")
    
    # Define explicit schema to avoid type conflicts
    schema = StructType([
        StructField("time", LongType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("open", DoubleType(), True),
        StructField("volumefrom", DoubleType(), True),
        StructField("volumeto", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("conversionType", StringType(), True),
        StructField("conversionSymbol", StringType(), True)
    ])
    
    # Create DataFrame with explicit schema
    df = spark.createDataFrame(raw_data, schema)
    
    # Add metadata columns for audit trail
    df_bronze = df \
        .withColumn("ingestion_timestamp", F.current_timestamp()) \
        .withColumn("source_system", F.lit("cryptocompare_api")) \
        .withColumn("data_quality_flag", F.lit("raw")) \
        .withColumn("ingestion_date", F.current_date())
    
    # Write to bronze table (append mode for incremental loads)
    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(bronze_table)
    
    record_count = df_bronze.count()
    print(f"\n✅ Ingested {record_count} records to {bronze_table}")
    print(f"📊 Time range: {datetime.fromtimestamp(raw_data[0]['time'])} to {datetime.fromtimestamp(raw_data[-1]['time'])}")
    
    return df_bronze

# Execute ingestion
df_bronze = ingest_to_bronze()

if df_bronze:
    print("\n" + "="*80)
    print("📋 Sample Bronze Data:")
    print("="*80)
    df_bronze.limit(5).show(truncate=False)

In [0]:
# Verify bronze table statistics
df_check = spark.table(bronze_table)

total_count = df_check.count()
distinct_time = df_check.select("time").distinct().count()
duplicates = total_count - distinct_time

print("=" * 80)
print("📊 BRONZE TABLE STATISTICS")
print("=" * 80)
print(f"Total Rows:      {total_count:,}")
print(f"Distinct Times:  {distinct_time:,}")
print(f"Duplicates:      {duplicates:,}")
print(f"\n💡 Duplicates are EXPECTED in Bronze layer (audit trail)")
print(f"   Silver layer will deduplicate to keep only latest records")
print("=" * 80)